# Tutorial: Pandas Functions Mastery Guide

## Outline

1. Generate realistic practice data
2. Inspect and clean DataFrames
3. Choose between `concat` and `merge`
4. Understand `groupby` objects and outputs
5. Create and manage multi-level headers
6. Reshape tables with `melt`, `pivot`, and `stack`

The notebook uses a single mini business scenario so every function feels connected.

In [1]:
from __future__ import annotations

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

SEED = 42
rng = np.random.default_rng(SEED)

print("pandas version:", pd.__version__)
print("Seed:", SEED)

# main functions: 
# - np.random.default_rng(seed)

pandas version: 3.0.1
Seed: 42


## Step 1 - Build deterministic random datasets

We create synthetic but realistic tables:
- `customers`: one row per customer
- `orders`: multiple rows per customer
- `monthly_targets`: department-level planning data
- `quarterly_sales_wide`: a wide table for reshaping practice

In [ ]:
n_customers = 18
customer_ids = np.arange(1001, 1001 + n_customers)

customers = pd.DataFrame(
{
"customer_id": customer_ids,
"segment": rng.choice(["Consumer", "Corporate", "Home Office"], size=n_customers, p=[0.5, 0.3, 0.2]),
"country": rng.choice(["France", "Germany", "Switzerland"], size=n_customers, p=[0.4, 0.35, 0.25]),
"signup_channel": rng.choice(["Web", "Partner", "Sales"], size=n_customers),
"loyalty_score": rng.integers(40, 100, size=n_customers),
}
).sort_values("customer_id").reset_index(drop=True)

n_orders = 90
orders = pd.DataFrame(
{
"order_id": np.arange(5001, 5001 + n_orders),
"customer_id": rng.choice(customer_ids, size=n_orders, replace=True),
"order_date": pd.to_datetime("2025-01-01") + pd.to_timedelta(rng.integers(0, 120, size=n_orders), unit="D"),
"product_family": rng.choice(["Analytics", "Automation", "Storage"], size=n_orders, p=[0.4, 0.35, 0.25]),
"region": rng.choice(["North", "South", "East", "West"], size=n_orders),
"quantity": rng.integers(1, 8, size=n_orders),
"unit_price": rng.choice([49, 79, 99, 149, 199], size=n_orders),
"discount_rate": rng.choice([0.0, 0.05, 0.10, 0.15], size=n_orders, p=[0.45, 0.25, 0.2, 0.1]),
}
)

orders["gross_revenue"] = orders["quantity"] * orders["unit_price"]
orders["net_revenue"] = orders["gross_revenue"] * (1 - orders["discount_rate"])
orders["order_month"] = orders["order_date"].dt.to_period("M").astype(str)

month_index = pd.period_range("2025-01", "2025-04", freq="M").astype(str)
region_list = ["North", "South", "East", "West"]
target_grid = pd.MultiIndex.from_product(
[month_index, region_list],
names=["order_month", "region"],
)
monthly_targets = target_grid.to_frame(index=False)
display(monthly_targets)
monthly_targets["target_revenue"] = rng.integers(2600, 3800, size=len(monthly_targets))

quarterly_sales_wide = pd.DataFrame(
{
"region": ["North", "South", "East", "West"],
"Q1": rng.integers(2200, 4000, size=4),
"Q2": rng.integers(2200, 4000, size=4),
"Q3": rng.integers(2200, 4000, size=4),
"Q4": rng.integers(2200, 4000, size=4),
}
)

display(customers.head())
display(orders.head())

# main functions: 
# - rng.integers(start, end, size)
# - rng.choice(['choices'], size, p)
# - pd.to_datetime('date'), pd.to_timedelta(value, unit)
# - np.arange(start, end, step)
# - table.sort_values('col', ascending) -> ascending is True by default
# - table['date'].dt.to_period()
# - pd.period_range(start, end, freq) -> gives the list of dates from start to end
# - pd.MultiIndex.from_product([lists], names=['name']) -> also from_tuples is common, one name per index level

,order_month,region
0,2025-01,North
1,2025-01,South
2,2025-01,East
3,2025-01,West
4,2025-02,North
5,2025-02,South
6,2025-02,East
7,2025-02,West
8,2025-03,North
9,2025-03,South


,customer_id,segment,country,signup_channel,loyalty_score
0,1001,Corporate,Switzerland,Partner,66
1,1002,Consumer,Germany,Web,88
2,1003,Home Office,Switzerland,Sales,90
3,1004,Corporate,France,Partner,63
4,1005,Consumer,Switzerland,Web,93


,order_id,customer_id,order_date,product_family,region,quantity,unit_price,discount_rate,gross_revenue,net_revenue,order_month
0,5001,1009,2025-02-10,Automation,North,6,199,0.10,1194,"1,074.60",2025-02
1,5002,1013,2025-04-26,Analytics,West,6,199,0.10,1194,"1,074.60",2025-04
2,5003,1005,2025-02-13,Analytics,East,4,99,0.00,396,396.00,2025-02
3,5004,1015,2025-04-20,Automation,West,1,149,0.00,149,149.00,2025-04
4,5005,1011,2025-03-01,Automation,South,4,79,0.00,316,316.00,2025-03


## Step 2 - Inspect the structure before doing anything fancy

Beginners often jump directly into `merge` or `groupby`, but pandas becomes much easier when you inspect:
- the shape
- the dtypes
- the meaning of the index
- the presence of missing values or duplicates

In [3]:
print("customers shape:", customers.shape)
print("orders shape:", orders.shape)

orders.info()

customers shape: (18, 5)
orders shape: (90, 11)
<class 'pandas.DataFrame'>
RangeIndex: 90 entries, 0 to 89
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   order_id        90 non-null     int64         
 1   customer_id     90 non-null     int64         
 2   order_date      90 non-null     datetime64[us]
 3   product_family  90 non-null     str           
 4   region          90 non-null     str           
 5   quantity        90 non-null     int64         
 6   unit_price      90 non-null     int64         
 7   discount_rate   90 non-null     float64       
 8   gross_revenue   90 non-null     int64         
 9   net_revenue     90 non-null     float64       
 10  order_month     90 non-null     str           
dtypes: datetime64[us](1), float64(2), int64(5), str(3)
memory usage: 7.9 KB


In [4]:
orders.describe(include="all").T[["count", "unique", "mean", "min", "max"]]

,count,unique,mean,min,max
order_id,90.00,NaN,"5,045.50","5,001.00","5,090.00"
customer_id,90.00,NaN,"1,009.10","1,001.00","1,018.00"
order_date,90,NaN,2025-02-27 03:12:00,2025-01-03 00:00:00,2025-04-29 00:00:00
product_family,90,3,NaN,NaN,NaN
region,90,4,NaN,NaN,NaN
quantity,90.00,NaN,4.08,1.00,7.00
unit_price,90.00,NaN,119.67,49.00,199.00
discount_rate,90.00,NaN,0.05,0.00,0.15
gross_revenue,90.00,NaN,488.81,49.00,"1,393.00"
net_revenue,90.00,NaN,461.69,46.55,"1,253.70"


In [5]:
# A quick quality check is often worth doing before analysis.
quality_checks = pd.DataFrame(
{
"missing_values": orders.isna().sum(),
"dtype": orders.dtypes.astype(str),
"n_unique": orders.nunique(),
}
)

quality_checks

# main functions: 
# - table.shape
# - table.info()
# - table.describe() -> usually use .T for readability and can select col of interest 
# - table.isna().sum() -> add a .sum() to get the tot amount over the entire table
# - table.dtypes 
# - table.nunique()

,missing_values,dtype,n_unique
order_id,0,int64,90
customer_id,0,int64,18
order_date,0,datetime64[us],65
product_family,0,str,3
region,0,str,4
quantity,0,int64,7
unit_price,0,int64,5
discount_rate,0,float64,4
gross_revenue,0,int64,30
net_revenue,0,float64,63


## Step 3 - Common row and column operations

Moves used every day:
- selecting columns
- filtering rows with boolean masks or `query`
- sorting
- creating derived columns with `assign`
- using `loc` when you want both row and column logic to stay explicit

In [6]:
selected = orders[["order_id", "customer_id", "region", "net_revenue"]].head(8)

high_value = orders.loc[
(orders["net_revenue"] > 500) & (orders["discount_rate"] <= 0.10),
["order_id", "region", "product_family", "net_revenue"],
].sort_values("net_revenue", ascending=False)

january_analytics = orders.query("order_month == '2025-01' and product_family == 'Analytics'")

selected, high_value.head(5), january_analytics.head(5)

(   order_id  customer_id region  net_revenue
 0      5001         1009  North     1,074.60
 1      5002         1013   West     1,074.60
 2      5003         1005   East       396.00
 3      5004         1015   West       149.00
 4      5005         1011  South       316.00
 5      5006         1009  South       398.00
 6      5007         1010   East       693.00
 7      5008         1011   West       168.30,
     order_id region product_family  net_revenue
 20      5021   West     Automation     1,253.70
 31      5032  South      Analytics     1,194.00
 84      5085   West     Automation     1,194.00
 37      5038  South        Storage     1,194.00
 0       5001  North     Automation     1,074.60,
     order_id  customer_id order_date product_family region  quantity  unit_price  discount_rate  gross_revenue  \
 18      5019         1002 2025-01-10      Analytics  South         7          99           0.15            693   
 19      5020         1014 2025-01-12      Analytics   East 

In [7]:
# assign returns a new DataFrame, which is handy in method chains
orders_enriched = (
orders
.assign(
revenue_band=lambda df: pd.cut(
df["net_revenue"],
bins=[0, 200, 500, 1000, np.inf],
labels=["Small", "Medium", "Large", "Enterprise"],
right=False,
),
is_discounted=lambda df: df["discount_rate"] > 0,
)
)

orders_enriched[["order_id", "net_revenue", "revenue_band", "is_discounted"]].head(10)

# main functions: 
# - table[['names']] -> to select a subset of col
# - table.iloc[x index, y index] -> compared to loc, it does not use names but positions
# - table.loc[cdts, ['names']] -> cdts like (table['col'] > xx) & (), can be () | () as well. and/or are not working here
# - table.query('query') -> SQL-like, ("col == value and col > value or col == 'str'")
# - table.assign(name=lambda df: what to do, another one if wanted) -> can be a bool for example, df['col] == value
# - pd.cut(table['col'], bins, labels) -> len(bins) = len(labels)+1

,order_id,net_revenue,revenue_band,is_discounted
0,5001,"1,074.60",Enterprise,True
1,5002,"1,074.60",Enterprise,True
2,5003,396.00,Medium,False
3,5004,149.00,Small,False
4,5005,316.00,Medium,False
5,5006,398.00,Medium,False
6,5007,693.00,Large,False
7,5008,168.30,Small,True
8,5009,298.00,Medium,False
9,5010,298.00,Medium,False


## Step 4 - `concat` versus `merge`: the mental model

This is one of the most important pandas distinctions.

Use `pd.concat(...)` when you want to glue objects together along an axis:
- stack rows from similar tables
- place columns side by side using the index

Use `DataFrame.merge(...)` when you want SQL-style matching based on key columns:
- enrich orders with customer attributes
- combine facts with lookup tables

Short rule:
- `concat` is about position and axis.
- `merge` is about key relationships.

In [8]:
jan_orders = orders.query("order_month == '2025-01'").copy()
feb_orders = orders.query("order_month == '2025-02'").copy()

stacked_orders = pd.concat([jan_orders, feb_orders], axis=0, ignore_index=True)

print("January rows:", len(jan_orders))
print("February rows:", len(feb_orders))
print("After vertical concat:", len(stacked_orders))

stacked_orders[["order_id", "order_month", "customer_id"]].head()

January rows: 21
February rows: 27
After vertical concat: 48


,order_id,order_month,customer_id
0,5019,2025-01,1002
1,5020,2025-01,1014
2,5023,2025-01,1011
3,5026,2025-01,1011
4,5032,2025-01,1008


In [9]:
# Horizontal concat aligns on the index, not on key columns.
customer_profile = customers.set_index("customer_id")[["segment", "country"]]
customer_scores = customers.set_index("customer_id")[["loyalty_score"]]

side_by_side = pd.concat([customer_profile, customer_scores], axis=1)
side_by_side.head()

,segment,country,loyalty_score
customer_id,,,
1001,Corporate,Switzerland,66
1002,Consumer,Germany,88
1003,Home Office,Switzerland,90
1004,Corporate,France,63
1005,Consumer,Switzerland,93


In [10]:
# merge matches rows by key columns. This is what you want for relational enrichment.
orders_with_customers = orders.merge(
customers,
on="customer_id",
how="left",
validate="many_to_one",
indicator=True,
)

orders_with_customers[["order_id", "customer_id", "segment", "country", "_merge"]].head()

,order_id,customer_id,segment,country,_merge
0,5001,1009,Consumer,Germany,both
1,5002,1013,Corporate,Germany,both
2,5003,1005,Consumer,Switzerland,both
3,5004,1015,Consumer,France,both
4,5005,1011,Consumer,France,both


### Why `merge` is not the same as horizontal `concat`

If two DataFrames share the same key values but not the same index values, horizontal `concat` can silently misalign rows.
`merge` prevents that by asking: "which rows match on this key?"

In [11]:
wrong_horizontal_concat = pd.concat(
[
orders[["order_id", "customer_id"]].head(5).reset_index(drop=True),
customers[["customer_id", "country"]].head(5).reset_index(drop=True),
],
axis=1,
)

correct_merge = (
orders[["order_id", "customer_id"]]
.head(5)
.merge(customers[["customer_id", "country"]], on="customer_id", how="left")
)

print("Horizontal concat can look valid while being logically wrong.")
wrong_horizontal_concat, correct_merge

# main functions: 
# - pd.concat([table1, table2], axis) -> axis is the contact line between both
# - table.set_index('col') -> set this col as index 
# - table1.merge(table2, on, how, validate, indicator) -> merge on a common col (on), avoid misalignment 

Horizontal concat can look valid while being logically wrong.


(   order_id  customer_id  customer_id      country
 0      5001         1009         1001  Switzerland
 1      5002         1013         1002      Germany
 2      5003         1005         1003  Switzerland
 3      5004         1015         1004       France
 4      5005         1011         1005  Switzerland,
    order_id  customer_id      country
 0      5001         1009      Germany
 1      5002         1013      Germany
 2      5003         1005  Switzerland
 3      5004         1015       France
 4      5005         1011       France)

## Step 5 - Important `merge` parameters

Parameters you will use constantly:
- `on`: shared key column(s)
- `left_on` / `right_on`: when the key names differ
- `how`: `left`, `right`, `inner`, `outer`
- `suffixes`: disambiguate overlapping column names
- `indicator=True`: show where rows came from
- `validate=...`: catch accidental row explosions

In [12]:
customer_lookup = customers.rename(columns={"customer_id": "cust_id"})

merged_with_different_key_names = orders.merge(
customer_lookup[["cust_id", "segment", "country"]],
left_on="customer_id",
right_on="cust_id",
how="left",
validate="many_to_one",
)

merged_with_different_key_names[["order_id", "customer_id", "cust_id", "segment"]].head()

,order_id,customer_id,cust_id,segment
0,5001,1009,1009,Consumer
1,5002,1013,1013,Corporate
2,5003,1005,1005,Consumer
3,5004,1015,1015,Consumer
4,5005,1011,1011,Consumer


In [13]:
# A tiny example to visualize the four join styles clearly.
left = pd.DataFrame({"key": ["A", "B", "C"], "left_value": [10, 20, 30]})
right = pd.DataFrame({"key": ["B", "C", "D"], "right_value": [200, 300, 400]})

join_examples = {
join_type: left.merge(right, on="key", how=join_type, indicator=True)
for join_type in ["inner", "left", "right", "outer"]
}

join_examples["inner"], join_examples["outer"], join_examples["left"], join_examples["right"]

(  key  left_value  right_value _merge
 0   B          20          200   both
 1   C          30          300   both,
   key  left_value  right_value      _merge
 0   A       10.00          NaN   left_only
 1   B       20.00       200.00        both
 2   C       30.00       300.00        both
 3   D         NaN       400.00  right_only,
   key  left_value  right_value     _merge
 0   A          10          NaN  left_only
 1   B          20       200.00       both
 2   C          30       300.00       both,
   key  left_value  right_value      _merge
 0   B       20.00          200        both
 1   C       30.00          300        both
 2   D         NaN          400  right_only)

In [ ]:
# validate is a safety belt. Here we create a duplicate key on purpose.
duplicate_customer_lookup = pd.concat(
[
customers[["customer_id", "segment"]],
customers[["customer_id", "segment"]].iloc[[0]],
],
ignore_index=True,
)

try:
    orders.merge(
    duplicate_customer_lookup,
    on="customer_id",
    how="left",
    validate="many_to_one",
    )
except Exception as exc:
    print(type(exc).__name__)
    print(exc)

# main functions: 
# - table.rename(columns={'prev':'new'})
# - merge param: how -> inner (if present in both), outer (all), left (both + table 1), right (both + table 2)
# - merge param: indicator -> show the origin of each row (left, right, both)

MergeError
Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
  customer_id
        1001 ...


## Step 6 - What does `groupby` actually return?

`groupby` does not immediately compute a final table.
It returns a lazy grouping object that remembers:
- the original data
- the grouping keys
- how rows are partitioned

You then ask that object for something:
- `.size()` for counts
- `.sum()` or `.mean()` for simple reductions
- `.agg(...)` for custom summary tables
- `.transform(...)` to return one value per original row
- `.filter(...)` to keep or discard full groups

In [15]:
grouped = orders.groupby("region")

print(type(grouped))
print("Group names:", list(grouped.groups.keys()))
print("Row indexes in North group:", grouped.groups["North"][:5])

<class 'pandas.api.typing.DataFrameGroupBy'>
Group names: ['East', 'North', 'South', 'West']
Row indexes in North group: Index([0, 9, 11, 17, 28], dtype='int64')


In [16]:
region_counts = grouped.size().rename("order_count")
region_revenue = grouped["net_revenue"].sum().rename("total_net_revenue")

region_counts, region_revenue

(region
 East     19
 North    22
 South    22
 West     27
 Name: order_count, dtype: int64,
 region
 East     9,001.15
 North    9,597.30
 South    9,955.80
 West    12,998.15
 Name: total_net_revenue, dtype: float64)

In [17]:
grouped_summary = (
orders
.groupby(["region", "product_family"], as_index=False)
.agg(
orders_count=("order_id", "count"),
avg_quantity=("quantity", "mean"),
total_revenue=("net_revenue", "sum"),
)
.sort_values(["region", "total_revenue"], ascending=[True, False])
)

grouped_summary.head(10)

,region,product_family,orders_count,avg_quantity,total_revenue
0,East,Analytics,9,4.78,"4,073.60"
2,East,Storage,6,4.83,"2,806.90"
1,East,Automation,4,4.25,"2,120.65"
4,North,Automation,9,4.11,"4,290.20"
3,North,Analytics,8,3.88,"2,849.50"
5,North,Storage,5,3.60,"2,457.60"
6,South,Analytics,9,4.44,"4,151.10"
7,South,Automation,7,4.43,"3,295.60"
8,South,Storage,6,4.17,"2,509.10"
10,West,Automation,8,4.50,"5,630.25"


In [18]:
# transform returns a Series aligned with the original rows.
# That makes it perfect for percentages, z-scores, and within-group normalization.
orders_with_share = orders.assign(
region_avg=lambda df: df.groupby("region")["net_revenue"].transform("mean"),
region_total=lambda df: df.groupby("region")["net_revenue"].transform("sum"),
pct_of_region=lambda df: df["net_revenue"] / df["region_total"],
)

orders_with_share[["order_id", "region", "net_revenue", "region_avg", "region_total", "pct_of_region"]].head()

,order_id,region,net_revenue,region_avg,region_total,pct_of_region
0,5001,North,"1,074.60",436.24,"9,597.30",0.11
1,5002,West,"1,074.60",481.41,"12,998.15",0.08
2,5003,East,396.00,473.74,"9,001.15",0.04
3,5004,West,149.00,481.41,"12,998.15",0.01
4,5005,South,316.00,452.54,"9,955.80",0.03


In [19]:
# filter keeps full groups when the condition is true for the group.
regions_with_large_average = orders.groupby("region").filter(
lambda group: group["net_revenue"].mean() > 455
)

regions_with_large_average[["order_id", "region", "net_revenue"]].head()

,order_id,region,net_revenue
1,5002,West,"1,074.60"
2,5003,East,396.00
3,5004,West,149.00
6,5007,East,693.00
7,5008,West,168.30


### `as_index=False` versus default behavior

Many people get confused because grouped columns become an index by default.

- Default: grouping keys become the index in the result.
- `as_index=False`: grouping keys stay as normal columns.

Both are valid. Choose the shape that makes the next step easiest.

In [ ]:
default_grouped = orders.groupby("region")["net_revenue"].sum()
flat_grouped = orders.groupby("region", as_index=False)["net_revenue"].sum()

print("Default result type:", type(default_grouped))
print("Flat result type:", type(flat_grouped))

default_grouped, flat_grouped

# main functions: 
# - table.groupby('col') -> not a table yet, used on columns with groups
# - table.groupby('col').groups.keys() -> gives the different groups present in this col
# - table.groupby('col').groups['name'] -> using the name of one of the groups, gives the index list of the elem within this group
# - table.groupby('col').size() -> gives a table with the number of elem in each group
# - table.groupby('col')['other_col'].sum() -> gives the sum of other_col per group 
# - table.groupby(['col1', 'col2']).agg(name=('col', 'method'), ...) -> looks for combinations of both, looks per combination, count-sum-mean for method
# - table.groupby('col')['other_col'].transform('method') -> unlike agg, keeps the original number of rows
# - table.groupby('col').filter(lambda group: group['other_col'].sum() < x) -> the group has to fill the condition to remain (differnt that row condition)
# - table.groupby('col', as_index=False) -> True by default, set the selected columns as index otherwise

Default result type: <class 'pandas.Series'>
Flat result type: <class 'pandas.DataFrame'>


(region
 East     9,001.15
 North    9,597.30
 South    9,955.80
 West    12,998.15
 Name: net_revenue, dtype: float64,
   region  net_revenue
 0   East     9,001.15
 1  North     9,597.30
 2  South     9,955.80
 3   West    12,998.15)

## Step 7 - Multi-level column headers from `agg`

Multi-level headers often appear after:
- `groupby(...).agg({...})`
- `pivot_table(...)`
- some reshaping operations

In [21]:
multi_header_summary = orders.groupby("region").agg(
{
"quantity": ["sum", "mean", "max"],
"net_revenue": ["sum", "mean", "max"],
}
)

multi_header_summary

quantity          net_revenue                
            sum mean max         sum   mean      max
region                                              
East         89 4.68   7    9,001.15 473.74 1,043.00
North        86 3.91   6    9,597.30 436.24 1,074.60
South        96 4.36   7    9,955.80 452.54 1,194.00
West         96 3.56   7   12,998.15 481.41 1,253.70

In [22]:
multi_header_summary.columns = multi_header_summary.columns.to_flat_index()
display(multi_header_summary)
multi_header_summary.columns = pd.MultiIndex.from_tuples(multi_header_summary.columns)
multi_header_summary

,"(quantity, sum)","(quantity, mean)","(quantity, max)","(net_revenue, sum)","(net_revenue, mean)","(net_revenue, max)"
region,,,,,,
East,89,4.68,7,"9,001.15",473.74,"1,043.00"
North,86,3.91,6,"9,597.30",436.24,"1,074.60"
South,96,4.36,7,"9,955.80",452.54,"1,194.00"
West,96,3.56,7,"12,998.15",481.41,"1,253.70"


quantity          net_revenue                
            sum mean max         sum   mean      max
region                                              
East         89 4.68   7    9,001.15 473.74 1,043.00
North        86 3.91   6    9,597.30 436.24 1,074.60
South        96 4.36   7    9,955.80 452.54 1,194.00
West         96 3.56   7   12,998.15 481.41 1,253.70

In [23]:
print("Column index object:", type(multi_header_summary.columns))
print("Column labels as tuples:", list(multi_header_summary.columns))

multi_header_summary[("net_revenue", "mean")]

Column index object: <class 'pandas.MultiIndex'>
Column labels as tuples: [('quantity', 'sum'), ('quantity', 'mean'), ('quantity', 'max'), ('net_revenue', 'sum'), ('net_revenue', 'mean'), ('net_revenue', 'max')]


region
East    473.74
North   436.24
South   452.54
West    481.41
Name: (net_revenue, mean), dtype: float64

In [24]:
# Flattening is common when you want easy downstream access or CSV export.
flattened = multi_header_summary.copy()
flattened.columns = ["_".join(col) for col in flattened.columns]
flattened = flattened.reset_index()

flattened.head()

# main functions: 
# - table.groupby('col').agg({'other_col': 'method'}) -> possible instead of name but we don't create a new name of our choice, can give a list of methods this way (MultiIndex then)
# - table.columns.to_flat_index() -> convert into tuple, combination of both index level, pd.MultiIndex.from_tuples(list) convert back 
# - table[(tuple)] -> to get a column in case of multiindex table, ('name level 1', 'name level 2')
# - ["_".join(col) for col in table.columns] -> instead of to_flat_index(), get a single name per column with _ join rather than tuples

,region,quantity_sum,quantity_mean,quantity_max,net_revenue_sum,net_revenue_mean,net_revenue_max
0,East,89,4.68,7,"9,001.15",473.74,"1,043.00"
1,North,86,3.91,6,"9,597.30",436.24,"1,074.60"
2,South,96,4.36,7,"9,955.80",452.54,"1,194.00"
3,West,96,3.56,7,"12,998.15",481.41,"1,253.70"


## Step 8 - MultiIndex rows and columns with pivot tables

`pivot_table` is a great bridge between raw transactional data and analysis tables.
It can create:
- multi-level row indexes
- multi-level column headers
- aggregated values in one step

In [25]:
revenue_pivot = pd.pivot_table(
orders,
index=["region", "product_family"],
columns="order_month",
values="net_revenue",
aggfunc="sum",
fill_value=0,
)

revenue_pivot

order_month            2025-01  2025-02  2025-03  2025-04
region product_family                                    
East   Analytics      1,686.15   910.50   894.00   582.95
       Automation         0.00     0.00   745.70 1,374.95
       Storage        1,658.20   693.00     0.00   455.70
North  Analytics      1,345.30   862.20     0.00   642.00
       Automation       376.20 2,042.75 1,273.05   598.20
       Storage          796.00 1,065.60   596.00     0.00
South  Analytics      1,783.05 1,169.70   906.80   291.55
       Automation       594.00 1,845.50   316.00   540.10
       Storage          495.00   420.75 1,593.35     0.00
West   Analytics      1,034.35 1,552.35 1,018.70 1,774.90
       Automation     1,074.60 1,253.70 1,968.90 1,333.05
       Storage          597.00 1,096.60   294.00     0.00

In [26]:
print("Row index type:", type(revenue_pivot.index))
print("First row index label:", revenue_pivot.index[0])
print("Column labels:", list(revenue_pivot.columns))

revenue_pivot.loc[("North", "Analytics")]

Row index type: <class 'pandas.MultiIndex'>
First row index label: ('East', 'Analytics')
Column labels: ['2025-01', '2025-02', '2025-03', '2025-04']


order_month
2025-01   1,345.30
2025-02     862.20
2025-03       0.00
2025-04     642.00
Name: (North, Analytics), dtype: float64

In [27]:
# stack moves columns into the index; unstack does the reverse.
pivot_stacked = revenue_pivot.stack().rename("revenue").reset_index()
pivot_stacked

# main functions:
# - pd.pivot_table(table, index=['col'], columns='other_col', values='third_col', aggfunc='method', fill_value) 
# -> if multiple col in index get multiindex, columns is the other we split into category, values is the one we analyze inside the table
# - table.loc[(tuple)] -> can be used for both a column or a row
# - pivot_table.stack() -> instead of grid of col and row (2 or more other categorical col), get every combination, one row for each

,region,product_family,order_month,revenue
0,East,Analytics,2025-01,"1,686.15"
1,East,Analytics,2025-02,910.50
2,East,Analytics,2025-03,894.00
3,East,Analytics,2025-04,582.95
4,East,Automation,2025-01,0.00
5,East,Automation,2025-02,0.00
6,East,Automation,2025-03,745.70
7,East,Automation,2025-04,"1,374.95"
8,East,Storage,2025-01,"1,658.20"
9,East,Storage,2025-02,693.00


## Step 9 - `melt`, `pivot`, and `pivot_table`

Use:
- `melt` to go from wide to long
- `pivot` to go from long to wide when each combination is unique
- `pivot_table` when duplicates exist and you need aggregation

In [28]:
quarterly_sales_wide

,region,Q1,Q2,Q3,Q4
0,North,2996,3030,3844,3952
1,South,3893,3115,3053,2441
2,East,2484,3443,2432,3906
3,West,2467,2927,2414,2700


In [29]:
quarterly_sales_long = quarterly_sales_wide.melt(
id_vars="region",
var_name="quarter",
value_name="sales",
)

quarterly_sales_long.head(8)

,region,quarter,sales
0,North,Q1,2996
1,South,Q1,3893
2,East,Q1,2484
3,West,Q1,2467
4,North,Q2,3030
5,South,Q2,3115
6,East,Q2,3443
7,West,Q2,2927


In [30]:
pivoted_back = quarterly_sales_long.pivot(index="region", columns="quarter", values="sales")
pivoted_back

quarter,Q1,Q2,Q3,Q4
region,,,,
East,2484,3443,2432,3906
North,2996,3030,3844,3952
South,3893,3115,3053,2441
West,2467,2927,2414,2700


In [31]:
display(quarterly_sales_long.iloc[[0]])

duplicated_long = pd.concat(
[
quarterly_sales_long,
quarterly_sales_long.iloc[[0]].assign(sales=quarterly_sales_long.iloc[0]["sales"] + 125),
],
ignore_index=True,
)

display(duplicated_long.head(2), duplicated_long.tail(2))

try:
    duplicated_long.pivot(index="region", columns="quarter", values="sales")
except Exception as exc:
    print(type(exc).__name__)
    print(exc)

# pivot_table works because it can aggregate duplicates.
duplicated_long.pivot_table(index="region", columns="quarter", values="sales", aggfunc="mean")

# main functions:
# - table.melt(id_vars='col', var_name='new_name', value_name='new_name') -> instead of grid one row per combination, rename the header with var_name and new col values with value_name
# - table.pivot(index='id_vars', columns='var_name', values='value_name') -> revert of melt, can't use if two row have the same grid but different value, pivot table with aggfunc can solve it

,region,quarter,sales
0,North,Q1,2996


,region,quarter,sales
0,North,Q1,2996
1,South,Q1,3893


,region,quarter,sales
15,West,Q4,2700
16,North,Q1,3121


ValueError
Index contains duplicate entries, cannot reshape


quarter,Q1,Q2,Q3,Q4
region,,,,
East,"2,484.00","3,443.00","2,432.00","3,906.00"
North,"3,058.50","3,030.00","3,844.00","3,952.00"
South,"3,893.00","3,115.00","3,053.00","2,441.00"
West,"2,467.00","2,927.00","2,414.00","2,700.00"


## Step 10 - Time-aware grouping and resampling-style thinking

Even without using full time-series APIs, pandas gets much nicer when you extract time parts explicitly.

In [32]:
monthly_region_summary = (
orders
.groupby(["order_month", "region"], as_index=False)
.agg(
total_orders=("order_id", "count"),
total_revenue=("net_revenue", "sum"),
avg_discount=("discount_rate", "mean"),
)
.sort_values(["order_month", "total_revenue"], ascending=[True, False])
)

monthly_region_summary

,order_month,region,total_orders,total_revenue,avg_discount
0,2025-01,East,6,"3,344.35",0.03
2,2025-01,South,4,"2,872.05",0.04
3,2025-01,West,5,"2,705.95",0.05
1,2025-01,North,6,"2,517.50",0.03
5,2025-02,North,8,"3,970.55",0.09
7,2025-02,West,8,"3,902.65",0.07
6,2025-02,South,7,"3,435.95",0.09
4,2025-02,East,4,"1,603.50",0.03
11,2025-03,West,9,"3,281.60",0.06
10,2025-03,South,8,"2,816.15",0.06


In [ ]:
# This merge compares actual revenue to the simple target table.
# Because the target table is keyed by month and region, this is a clean many-to-one merge.
actual_vs_target = monthly_region_summary.merge(
monthly_targets,
on=["order_month", "region"],
how="left",
validate="many_to_one",
)

actual_vs_target["gap_to_target_template"] = (
actual_vs_target["total_revenue"] - actual_vs_target["target_revenue"]
)

actual_vs_target.head(8)

# main functions:
# - table.sort_values(['sol1', 'col2'], ascending=[True, False]) -> can use with multiple col and different rules
# - table.merge(table2, on=['col1', 'col2']) -> can also work on multiple col 

,order_month,region,total_orders,total_revenue,avg_discount,target_revenue,gap_to_target_template
0,2025-01,East,6,"3,344.35",0.03,3647,-302.65
1,2025-01,South,4,"2,872.05",0.04,3363,-490.95
2,2025-01,West,5,"2,705.95",0.05,3286,-580.05
3,2025-01,North,6,"2,517.50",0.03,2905,-387.50
4,2025-02,North,8,"3,970.55",0.09,3135,835.55
5,2025-02,West,8,"3,902.65",0.07,3735,167.65
6,2025-02,South,7,"3,435.95",0.09,2774,661.95
7,2025-02,East,4,"1,603.50",0.03,2897,"-1,293.50"


## Step 11 - Missing values, duplicates, and safe habits

Real data is messy, so here are a few reliable habits:
- check `isna().sum()`
- check duplicate keys with `.duplicated(...)`
- use `validate=` in merges
- prefer explicit column selection over "I hope this works"

In [34]:
messy_orders = orders.copy()
display(messy_orders.sample(3, random_state=SEED))

,order_id,customer_id,order_date,product_family,region,quantity,unit_price,discount_rate,gross_revenue,net_revenue,order_month
40,5041,1015,2025-02-07,Storage,South,5,99,0.15,495,420.75,2025-02
22,5023,1011,2025-01-16,Storage,West,1,199,0.00,199,199.00,2025-01
55,5056,1008,2025-02-09,Analytics,West,3,49,0.05,147,139.65,2025-02


In [35]:
messy_orders.loc[messy_orders.sample(3, random_state=SEED).index, "discount_rate"] = np.nan
messy_orders = pd.concat([messy_orders, messy_orders.iloc[[0]]], ignore_index=True)

pd.DataFrame(
{
"missing_values": messy_orders.isna().sum(),
"duplicate_order_ids": [messy_orders["order_id"].duplicated().sum()] * len(messy_orders.columns),
},
index=messy_orders.columns,
)

,missing_values,duplicate_order_ids
order_id,0,1
customer_id,0,1
order_date,0,1
product_family,0,1
region,0,1
quantity,0,1
unit_price,0,1
discount_rate,3,1
gross_revenue,0,1
net_revenue,0,1


In [ ]:
cleaned_orders = (
messy_orders
.drop_duplicates(subset="order_id")
.assign(discount_rate=lambda df: df["discount_rate"].fillna(0))
)

len(messy_orders), len(cleaned_orders), cleaned_orders["discount_rate"].isna().sum()

# main functions:
# - table.sample(n, random_state)
# - table['col'].duplicated() -> gives a serie of bool with True if a value appeared before
# - table.drop_duplicates(subset='col') -> drop duplicates (keep the first) of the cols in subset
# - table['col'].fillna(value) -> fill nan by a value 

(91, 90, np.int64(0))

## Step 12 - A small end-to-end pipeline

This kind of pipeline is where pandas starts to feel elegant:
1. merge lookup data
2. create derived columns
3. group and aggregate
4. sort for reporting

In [37]:
final_report = (
orders
.merge(customers[["customer_id", "segment", "country"]], on="customer_id", how="left")
.assign(
discount_flag=lambda df: np.where(df["discount_rate"] > 0, "Discounted", "Full price"),
month=lambda df: df["order_date"].dt.to_period("M").astype(str),
)
.groupby(["month", "segment", "discount_flag"], as_index=False)
.agg(
orders_count=("order_id", "count"),
customers_count=("customer_id", "nunique"),
revenue=("net_revenue", "sum"),
)
.sort_values(["month", "revenue"], ascending=[True, False])
)

final_report.head(12)

# that's where it becomes useful to use methods in a row

,month,segment,discount_flag,orders_count,customers_count,revenue
0,2025-01,Consumer,Discounted,5,4,"2,800.65"
1,2025-01,Consumer,Full price,6,5,"2,531.00"
5,2025-01,Home Office,Full price,3,3,"2,155.00"
2,2025-01,Corporate,Discounted,3,2,"1,729.90"
3,2025-01,Corporate,Full price,3,2,"1,686.00"
4,2025-01,Home Office,Discounted,1,1,537.30
6,2025-02,Consumer,Discounted,10,6,"5,625.80"
7,2025-02,Consumer,Full price,6,5,"2,632.00"
10,2025-02,Home Office,Discounted,5,4,"2,381.00"
8,2025-02,Corporate,Discounted,4,2,"1,383.85"


## Functions recap
---
- np.random.default_rng(seed)
---
- rng.integers(start, end, size)
- rng.choice(['choices'], size, p)
- pd.to_datetime('date'), pd.to_timedelta(value, unit)
- np.arange(start, end, step)
- table.sort_values('col', ascending) -> ascending is True by default
- table['date'].dt.to_period()
- pd.period_range(start, end, freq) -> gives the list of dates from start to end
- pd.MultiIndex.from_product([lists], names=['name']) -> also from_tuples is common, one name per index level
---
- table.shape
- table.info()
- table.describe() -> usually use .T for readability and can select col of interest 
- table.isna().sum() -> add a .sum() to get the tot amount over the entire table
- table.dtypes 
- table.nunique()
---
- table[['names']] -> to select a subset of col
- table.iloc[x index, y index] -> compared to loc, it does not use names but positions
- table.loc[cdts, ['names']] -> cdts like (table['col'] > xx) & (), can be () | () as well. and/or are not working here
- table.query('query') -> SQL-like, ("col == value and col > value or col == 'str'")
- table.assign(name=lambda df: what to do, another one if wanted) -> can be a bool for example, df['col] == value
- pd.cut(table['col'], bins, labels) -> len(bins) = len(labels)+1
---
- pd.concat([table1, table2], axis) -> axis is the contact line between both
- table.set_index('col') -> set this col as index 
- table1.merge(table2, on, how, validate, indicator) -> merge on a common col (on), avoid misalignment 
---
- table.rename(columns={'prev':'new'})
- merge param: how -> inner (if present in both), outer (all), left (both + table 1), right (both + table 2)
- merge param: indicator -> show the origin of each row (left, right, both)
---
- table.groupby('col') -> not a table yet, used on columns with groups
- table.groupby('col').groups.keys() -> gives the different groups present in this col
- table.groupby('col').groups['name'] -> using the name of one of the groups, gives the index list of the elem within this group
- table.groupby('col').size() -> gives a table with the number of elem in each group
- table.groupby('col')['other_col'].sum() -> gives the sum of other_col per group 
- table.groupby(['col1', 'col2']).agg(name=('col', 'method'), ...) -> looks for combinations of both, looks per combination, count-sum-mean for method
- table.groupby('col')['other_col'].transform('method') -> unlike agg, keeps the original number of rows
- table.groupby('col').filter(lambda group: group['other_col'].sum() < x) -> the group has to fill the condition to remain (differnt that row condition)
- table.groupby('col', as_index=False) -> True by default, set the selected columns as index otherwise
---
- table.groupby('col').agg({'other_col': 'method'}) -> possible instead of name but we don't create a new name of our choice, can give a list of methods this way (MultiIndex then)
- table.columns.to_flat_index() -> convert into tuple, combination of both index level, pd.MultiIndex.from_tuples(list) convert back 
- table[(tuple)] -> to get a column in case of multiindex table, ('name level 1', 'name level 2')
- ["_".join(col) for col in table.columns] -> instead of to_flat_index(), get a single name per column with _ join rather than tuples
---
- pd.pivot_table(table, index=['col'], columns='other_col', values='third_col', aggfunc='method', fill_value) 
-> if multiple col in index get multiindex, columns is the other we split into category, values is the one we analyze inside the table
- table.loc[(tuple)] -> can be used for both a column or a row
- pivot_table.stack() -> instead of grid of col and row (2 or more other categorical col), get every combination, one row for each
---
- table.melt(id_vars='col', var_name='new_name', value_name='new_name') -> instead of grid one row per combination, rename the header with var_name and new col values with value_name
- table.pivot(index='id_vars', columns='var_name', values='value_name') -> revert of melt, can't use if two row have the same grid but different value, pivot table with aggfunc can solve it
---
- table.sort_values(['sol1', 'col2'], ascending=[True, False]) -> can use with multiple col and different rules
- table.merge(table2, on=['col1', 'col2']) -> can also work on multiple col 
---
- table.sample(n, random_state)
- table['col'].duplicated() -> gives a serie of bool with True if a value appeared before
- table.drop_duplicates(subset='col') -> drop duplicates (keep the first) of the cols in subset
- table['col'].fillna(value) -> fill nan by a value 
---